# Welcome to Open Source ML Platform

This notebook demonstrates the capabilities of the 100% open-source ML platform.

## Available Services:

- **JupyterLab**: http://localhost:8888 (this notebook)
- **VS Code Server**: http://localhost:8443
- **MLflow**: http://localhost:5000
- **Apache Airflow**: http://localhost:8080 (admin/admin)
- **Grafana**: http://localhost:3001 (admin/admin)
- **MinIO Console**: http://localhost:9001 (minioadmin/minioadmin)
- **Prometheus**: http://localhost:9090
- **BentoML**: http://localhost:3000
- **Ollama API**: http://localhost:11434

## 1. Environment Check

In [ ]:
import sys
import os

print(f"Python Version: {sys.version}")
print(f"\nEnvironment Variables:")
print(f"MLFLOW_TRACKING_URI: {os.getenv('MLFLOW_TRACKING_URI')}")
print(f"OLLAMA_HOST: {os.getenv('OLLAMA_HOST')}")
print(f"MINIO_ENDPOINT: {os.getenv('MINIO_ENDPOINT')}")

## 2. Test PyTorch

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 3. Test TensorFlow

In [ ]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

## 4. Test MLflow Connection

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://mlflow:5000")
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# Create a test experiment
experiment_name = "test_experiment"
mlflow.set_experiment(experiment_name)

with mlflow.start_run():
    mlflow.log_param("test_param", "hello_mlflow")
    mlflow.log_metric("test_metric", 0.95)
    print("✓ MLflow logging successful!")

## 5. Test MinIO (S3) Connection

In [ ]:
import boto3
from botocore.client import Config

# Create S3 client for MinIO
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# List buckets
response = s3_client.list_buckets()
print("Available buckets:")
for bucket in response['Buckets']:
    print(f"  - {bucket['Name']}")

## 6. Test Ollama (Local LLM)

In [ ]:
import requests
import json

# Test Ollama API
ollama_url = "http://ollama:11434/api/tags"

try:
    response = requests.get(ollama_url)
    models = response.json()
    print("Available Ollama models:")
    for model in models.get('models', []):
        print(f"  - {model['name']}")
except Exception as e:
    print(f"Error connecting to Ollama: {e}")

## 7. Simple ML Example with Scikit-learn

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import mlflow
import mlflow.sklearn

# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Train with MLflow tracking
mlflow.set_experiment("iris_classification")

with mlflow.start_run():
    # Train model
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    # Evaluate
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Log to MLflow
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(clf, "model")
    
    print(f"✓ Model trained with accuracy: {accuracy:.4f}")
    print(f"✓ Model logged to MLflow")

## 8. Test LangChain with Local Ollama

In [ ]:
try:
    from langchain.llms import Ollama
    from langchain.prompts import PromptTemplate
    from langchain.chains import LLMChain
    
    # Initialize Ollama LLM
    llm = Ollama(
        base_url="http://ollama:11434",
        model="mistral"
    )
    
    # Create a simple chain
    template = """Question: {question}
    
    Answer: Let me help you with that."""
    
    prompt = PromptTemplate(template=template, input_variables=["question"])
    chain = LLMChain(prompt=prompt, llm=llm)
    
    # Test
    response = chain.run("What is machine learning?")
    print(f"LangChain + Ollama Response:\n{response}")
except Exception as e:
    print(f"LangChain test skipped (install langchain if needed): {e}")

## 9. Data Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create sample data
x = np.linspace(0, 10, 100)
y1 = np.sin(x)
y2 = np.cos(x)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(x, y1, label='sin(x)', linewidth=2)
plt.plot(x, y2, label='cos(x)', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Trigonometric Functions')
plt.legend()
plt.grid(True)
plt.show()

## 10. Summary

This notebook demonstrated:

✅ PyTorch and TensorFlow integration  
✅ MLflow experiment tracking  
✅ MinIO (S3-compatible) storage  
✅ Ollama local LLM inference  
✅ Scikit-learn ML pipeline  
✅ LangChain integration  
✅ Data visualization  

### Next Steps:

1. Explore MLflow UI: http://localhost:5000
2. Create Airflow DAGs: http://localhost:8080
3. Monitor with Grafana: http://localhost:3001
4. Use MinIO for data storage: http://localhost:9001
5. Experiment with local LLMs via Ollama

### Useful Commands:

```bash
# Check service status
docker-compose ps

# View logs
docker-compose logs -f jupyter

# Stop all services
docker-compose down

# Restart a service
docker-compose restart mlflow
```